In [1]:
import pandas as pd
X_new = pd.read_csv('/Users/jingyuan/Documents/ChatGPT/模型构建/skinaging_predictor_ZJU/Database/X_320_clean_LR.csv', delimiter=',',header = None,index_col= 0, skiprows=1)
#X_out = pd.read_csv('generalization_data/X_test_nr_320.csv',delimiter=',',header = None,index_col= 0, skiprows=1)
#X_out_Sequence = pd.read_csv('generalization_data/X_test_nr.csv',delimiter=',',header = None,index_col= 0, dtype={0: int},skiprows=1)
# load numpy array from csv file
from numpy import loadtxt
import numpy as np
# load array
y_new = loadtxt('/Users/jingyuan/Documents/ChatGPT/模型构建/skinaging_predictor_ZJU/Database/y_after_clean.csv', delimiter=',')



In [2]:
X_new.head()

,1,2,3,4,5,6,7,8,9,10,...,311,312,313,314,315,316,317,318,319,320
0,,,,,,,,,,,,,,,,,,,,,
0,0.139993,-0.435624,0.533432,0.245520,-0.037777,-0.205565,-0.174132,0.006184,-0.147792,-0.072895,...,0.188533,0.027038,-0.072476,-0.008486,-0.004960,0.064057,-0.081112,-0.069016,0.309268,0.055681
1,-0.017159,-0.274750,0.139635,0.226395,0.091277,-0.054095,-0.197418,-0.121457,-0.167074,-0.133363,...,0.266903,0.042168,0.221770,0.223856,-0.075128,0.288996,0.195563,0.168286,0.241382,0.130343
3,0.081766,-0.434443,0.517479,0.393442,-0.139180,-0.236886,-0.208077,0.020925,-0.149324,-0.134162,...,-0.059169,0.134012,0.071508,0.094889,0.037986,0.362630,0.021677,0.007231,0.223266,0.109919
5,0.046700,-0.324264,0.292027,0.163590,-0.017439,-0.094095,-0.126638,-0.081752,-0.165306,-0.072302,...,0.203659,-0.086773,0.235853,0.144662,-0.110621,0.074933,0.060307,0.075304,0.219589,0.072603
6,0.098760,-0.232188,0.205257,0.144129,-0.021316,0.049918,-0.277318,-0.154654,-0.208609,-0.151379,...,0.074428,0.049859,0.178146,-0.044487,-0.079229,0.000859,-0.003936,0.048288,0.290824,0.020416


In [3]:
import sys
sys.path.insert(0,'/Users/jingyuan/Documents/ChatGPT/模型构建/skinaging_predictor_ZJU')
import statistics
from sklearn.metrics import balanced_accuracy_score
from self_function import evaluation as eva

In [4]:
# dataset splitting 
from sklearn.model_selection import train_test_split
X_train_whole, X_ind_test, y_train_whole, y_ind_test =  train_test_split(X_new, y_new, test_size=0.2, random_state=2001)

In [5]:
# logistic regresion without penality and solver
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
clf = LogisticRegression(max_iter = 5000)
param_grid = {'C':[0.5,1,1.5,2,2.5,4,6,8,13,14,15,16,20,30,40, 50,60,70, 80, 90 ,100,125, 500, 1000,2000,4000,10000]}
grid_search = GridSearchCV(clf,param_grid,cv=10,n_jobs=-1)
grid_search.fit(X_train_whole,y_train_whole)
best_model_reg = grid_search.best_estimator_
y_pred = best_model_reg.predict(X_out)
#model performance evaluation:BACC & recall & accuracy & MCC & f1 score& roc_auc

evaluation = eva(best_model_reg,X_new,y_new)
print(evaluation)

#print the model's parameters and validation score
print("About model development")
print("Best Parameters:{}".format(grid_search.best_params_))
print("Best cross_validation socre:{:.3f}".format(grid_search.best_score_))
print("Test set score:{:.2f}".format(grid_search.score(X_ind_test,y_ind_test)))
print("Best_estimator:\n{}".format(grid_search.best_estimator_))

# get the probability of class 1
y_pred_prob = best_model_reg.predict_proba(X_out)[:, 1]

# 根据排序后的索引获取预测结果和概率
sorted_indices = np.argsort(y_pred_prob)[::-1]
sorted_y_pred = y_pred[sorted_indices]
sorted_prob = y_pred_prob[sorted_indices]

sorted_sequence = X_out_Sequence.iloc[sorted_indices]

# 将预测结果为0的序列移到最后
zero_indices = np.where(sorted_y_pred == 0)[0]
non_zero_indices = np.where(sorted_y_pred != 0)[0]
sorted_sequence = pd.concat([sorted_sequence.iloc[non_zero_indices], sorted_sequence.iloc[zero_indices]])
sorted_y_pred = np.concatenate((sorted_y_pred[non_zero_indices], sorted_y_pred[zero_indices]))
sorted_prob = np.concatenate((sorted_prob[non_zero_indices], sorted_prob[zero_indices]))

# 输出结果
output = np.column_stack((sorted_sequence, sorted_y_pred, sorted_prob))

print("Sorted Sequence, Predictions, and Probabilities:")
for item in output:
    if item[2] == 0:
        print(item[0], item[1], item[2],'-')
    else:
        print(item[0], item[1], item[2],item[3])



NameError: name 'X_out' is not defined

In [ ]:
#random forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
clf = RandomForestClassifier(n_jobs=-1)
param_grid = {'n_estimators': [80, 160, 320, 480, 640, 1280,2000],
            'max_depth': [1, 2, 3, 4,5],
            'max_features' :['log2', 'sqrt']}
grid_search = GridSearchCV(clf,param_grid,cv=10,n_jobs=-1)
grid_search.fit(X_train_whole,y_train_whole)
best_model_reg = grid_search.best_estimator_
#y_pred = best_model_reg.predict(X_out)
#评价 准确率 recall 精确率 rocauc f1 mcc

evaluation = eva(best_model_reg,X_new,y_new)
print(evaluation)

#print the model's parameters and validation score
print("About model development")
print("Best Parameters:{}".format(grid_search.best_params_))
print("Best cross_validation socre:{:.3f}".format(grid_search.best_score_))
print("Test set score:{:.2f}".format(grid_search.score(X_ind_test,y_ind_test)))
print("Best_estimator:\n{}".format(grid_search.best_estimator_))


In [ ]:
# KNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
clf = KNeighborsClassifier(n_jobs=-1)
param_grid = {'n_neighbors' : [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43],'weights' : ['uniform', 'distance'],
              'algorithm' : ['auto', 'ball_tree', 'kd_tree', 'brute'],
              'leaf_size' : [10, 20, 30, 40, 50, 60, 70, 80]}
grid_search = GridSearchCV(clf,param_grid,cv=10,n_jobs=-1)
grid_search.fit(X_train_whole,y_train_whole)
best_model_reg = grid_search.best_estimator_
#y_pred = best_model_reg.predict(X_out)
#评价 准确率 recall 精确率 rocauc f1 mcc

evaluation = eva(best_model_reg,X_new,y_new)
print(evaluation)

#print the model's parameters and validation score
print("About model development")
print("Best Parameters:{}".format(grid_search.best_params_))
print("Best cross_validation socre:{:.3f}".format(grid_search.best_score_))
print("Test set score:{:.2f}".format(grid_search.score(X_ind_test,y_ind_test)))
print("Best_estimator:\n{}".format(grid_search.best_estimator_))



In [ ]:
# SVM
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
clf = SVC()
param_grid = {'C':[0.001, 0.01, 0.1,  0.5, 1.1, 1.3, 5, 10, 30, 50, 100, 300],'kernel':['linear', 'poly', 'rbf', 'sigmoid'],
              'degree':[1, 3, 5, 7, 9],'tol':[1e-5]}
grid_search = GridSearchCV(clf,param_grid,cv=10,n_jobs=-1)
grid_search.fit(X_train_whole,y_train_whole)
best_model_reg = grid_search.best_estimator_
#y_pred = best_model_reg.predict(X_out)
#评价 准确率 recall 精确率 rocauc f1 mcc

evaluation = eva(best_model_reg,X_new,y_new)
print(evaluation)

#save model
import pickle


#print the model's parameters and validation score
print("About model development")
print("Best Parameters:{}".format(grid_search.best_params_))
print("Best cross_validation socre:{:.3f}".format(grid_search.best_score_))
print("Test set score:{:.2f}".format(grid_search.score(X_ind_test,y_ind_test)))
print("Best_estimator:\n{}".format(grid_search.best_estimator_))



In [ ]:
# LightGBM
import lightgbm as lgb
import itertools
from sklearn.model_selection import GridSearchCV


clf = lgb.LGBMClassifier(boosting='gbdt', objective='binary',n_jobs=-1,verbose = -1)
param_grid = {'learning_rate':[0.001,0.01,0.1,0.2,0.5,1],
              'n_estimators':[40,80,160,320,640],
              'max_depth':[-1],
              'reg_lambda':[0]}
grid_search = GridSearchCV(clf,param_grid,cv=10,n_jobs=-1)
grid_search.fit(X_train_whole,y_train_whole)
best_model_reg = grid_search.best_estimator_
#y_pred = best_model_reg.predict(X_out)


#print the model's parameters and validation score
print("About model development")
print("Best Parameters:{}".format(grid_search.best_params_))
print("Best cross_validation socre:{:.3f}".format(grid_search.best_score_))
print("Test set score:{:.2f}".format(grid_search.score(X_ind_test,y_ind_test)))
print("Best_estimator:\n{}".format(grid_search.best_estimator_))

#model performance evaluation:BACC & recall & accuracy & MCC & f1 score& roc_auc
evaluation = eva(best_model_reg,X_new,y_new)
print(evaluation)




## DeepLearning深度学习

In [ ]:
#get data
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense,BatchNormalization,Dropout
from numpy import loadtxt
from sklearn.metrics import classification_report, roc_auc_score, matthews_corrcoef
import keras

# 读取数据
data = X_new
# 将数据分为特征和标签
X = data.iloc[:, 0:].values
print(X.shape)
y = y_new
print(y.shape)


In [ ]:
#RNN
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, matthews_corrcoef
from keras.models import Sequential
from keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import SGD



# 数据预处理
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
input_shape = (1, X_train.shape[2])

# 构建RNN模型
model = Sequential()
model = Sequential()
model.add(LSTM(128, input_shape=input_shape, return_sequences=True))
model.add(LSTM(64, return_sequences=True))
model.add(LSTM(32,return_sequences = True))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.15))
model.add(Dense(10, activation='relu'))
model.add(Dropout(0.15))
model.add(Dense(1, activation='sigmoid'))



# 编译模型
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['binary_accuracy'])

# 训练模型
model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1)

# 在测试集上评估模型
y_pred = model.predict(X_test)
y_pred_proba = y_pred.flatten()
y_pred_binary = np.round(y_pred_proba)

print("Classification Report:")
print(classification_report(y_test, y_pred_binary))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print("ROC AUC:", roc_auc)

mcc = matthews_corrcoef(y_test, y_pred_binary)
print("MCC:", mcc)



